In [ ]:
# This notebook applies Resampling methods

###
# 0. Preparation
###
# Importing libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

from sklearn.metrics import classification_report, f1_score
from sklearn import linear_model, preprocessing
from sklearn.model_selection import train_test_split
from sklearn import svm, neighbors

In [ ]:
# Mounting GoogleDrive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Reading data file from GoogleDrive
df = pd.read_pickle("/content/drive/My Drive/Data Science/Team Project X-Rays/Dataframes/df_basic.pkl")
df.head()

# Define random subsample for computation efficiency
#df = df.sample(500)



,Name,Case,PX_1,PX_2,PX_3,PX_4,PX_5,PX_6,PX_7,PX_8,...,PX_4087,PX_4088,PX_4089,PX_4090,PX_4091,PX_4092,PX_4093,PX_4094,PX_4095,PX_4096
0,COVID-122,COVID,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.968627,0.956863,0.945098,0.925490,0.905882,0.894118,0.882353,0.854902,0.839216,0.788235
1,COVID-1689,COVID,0.203922,0.050980,0.043137,0.039216,0.039216,0.043137,0.070588,0.054902,...,0.807843,0.803922,0.733333,0.654902,0.600000,0.513725,0.427451,0.164706,0.172549,0.172549
2,COVID-2636,COVID,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.756863,0.745098,0.737255,0.752941,0.737255,0.713725,0.690196,0.686275,0.647059,0.631373
3,COVID-3007,COVID,0.074510,0.050980,0.027451,0.031373,0.031373,0.019608,0.003922,0.133333,...,0.768627,0.764706,0.752941,0.752941,0.650980,0.619608,0.156863,0.000000,0.000000,0.000000
4,COVID-1620,COVID,0.152941,0.133333,0.133333,0.133333,0.133333,0.137255,0.129412,0.133333,...,0.517647,0.498039,0.498039,0.470588,0.423529,0.364706,0.423529,0.486275,0.415686,0.317647


In [ ]:
# 1. Data preprocessing
###

# Create categorical variable from Case
df["Case"] = df.Case.replace({"Normal": 0, "COVID": 1, "Lung_Opacity": 2, "Viral Pneumonia": 3})
df.Case.astype(int)

# Check construction
df.Case.value_counts()

# Split data into target and features
target = df.Case

# Features data: Drop Names and target
data = df.drop(["Name", "Case"], axis = 1)
data.head()
data.shape

# Split data into training and Test sets, save random state
X_train, X_test, y_train, y_test = train_test_split(data, target, test_size = 0.2, random_state = 123)

In [ ]:
# 4. Resampling Methods
###
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler, ClusterCentroids

# Display precentage distribution of target variable
target.value_counts(normalize = True)

# => It's not that unbalanced, except for class 3 (Viral Pneumonia) at only 6.3%

,proportion
Case,
0,0.482871
2,0.284861
1,0.168870
3,0.063397


In [ ]:
# 4.1 Undersampling
###

import time
start_time = time.time()

# Random Undersampling
rUs = RandomUnderSampler()

# Create new undersampled data sets
X_ru, y_ru = rUs.fit_resample(X_train, y_train)

# Show number of classes in resulting sample
print("Classes in full sample:", dict(pd.Series(y_train).value_counts()))
print("Classes in random undersample:", dict(pd.Series(y_ru).value_counts()))



Classes in full sample: {0: 8088, 2: 4847, 1: 2883, 3: 1066}
Classes in random undersample: {0: 1066, 1: 1066, 2: 1066, 3: 1066}


In [ ]:
# Model 5 - rUs with SVM
###
clf5_name = "rUs with SM"
clf5 = svm.SVC(gamma = 0.01, kernel = "poly")

# Train the model on the randomoversampler training sets
clf5.fit(X_ru, y_ru)

# Make predictions
y_pred = clf5.predict(X_test)

# Calc accuracys
clf5_score = clf5.score(X_test, y_test)

# Calc mean unweighted F1 Score in all classes
clf5_f1 = f1_score(y_test, y_pred, average = "macro")

# Measure time
model5_time = (time.time() - start_time)/60



In [ ]:
# Show Results
###
from imblearn.metrics import classification_report_imbalanced, geometric_mean_score

# Modelling Time
print("Model 5: --- %s minutes ---" % model5_time)

# Score and F1-Score
print("The score is:", clf5_score)
print("The mean F1-Score (unweighted) is:", clf5_f1)

# Show Confusion Matrix
cm5 = pd.crosstab(y_test, y_pred, rownames = ['Realised Class'], colnames = ['Predicted Class'])
display(cm5)

# Show classification report
model5_cr = classification_report(y_test, y_pred)
print(model5_cr)

Model 5: --- 1.5326174298922222 minutes ---
The score is: 0.7427149964463398
The mean F1-Score (unweighted) is: 0.7462149628627308


Predicted Class,0,1,2,3
Realised Class,,,,
0,1566,225,243,69
1,98,492,85,6
2,166,171,812,16
3,2,3,2,265


              precision    recall  f1-score   support

           0       0.85      0.74      0.80      2103
           1       0.55      0.72      0.63       681
           2       0.71      0.70      0.70      1165
           3       0.74      0.97      0.84       272

    accuracy                           0.74      4221
   macro avg       0.72      0.78      0.74      4221
weighted avg       0.76      0.74      0.75      4221



In [ ]:
# 4.2. Resampling with Class weights: Changing the class weights to penalize errors on classes
# linear SVM
###
import time
start_time = time.time()

# Instantiate SVM
clf6_name = "linear SVM with class weights"
clf6 = svm.SVC(gamma = 0.01, kernel = "poly", class_weight = "balanced")

# Train the model on training data
clf6.fit(X_train, y_train)

# Make predictions on test set
y_pred = clf6.predict(X_test)

# Calc accuracys
clf6_score = clf6.score(X_test, y_test)

# Calc mean unweighted F1 Score in all classes
clf6_f1 = f1_score(y_test, y_pred, average = "macro")

# Measure time
model6_time = (time.time() - start_time)/60


In [ ]:
# Show Results
###

# Modelling Time
print("Model 6: --- %s minutes ---" % model6_time)

# Score and F1-Score
print("The score is:", clf6_score)
print("The mean F1-Score (unweighted) is:", clf6_f1)

# Show Confusion Matrix
cm6 = pd.crosstab(y_test, y_pred, rownames = ['Realised Class'], colnames = ['Predicted Class'])
display(cm6)

# Show classification report
model6_cr = classification_report(y_test, y_pred)
print(model6_cr)

Model 6: --- 16.081615642706552 minutes ---
The score is: 0.8012319355602938
The mean F1-Score (unweighted) is: 0.8010986756990126


Predicted Class,0,1,2,3
Realised Class,,,,
0,1780,103,200,20
1,121,478,81,1
2,199,76,886,4
3,20,5,9,238


              precision    recall  f1-score   support

           0       0.84      0.85      0.84      2103
           1       0.72      0.70      0.71       681
           2       0.75      0.76      0.76      1165
           3       0.90      0.88      0.89       272

    accuracy                           0.80      4221
   macro avg       0.81      0.80      0.80      4221
weighted avg       0.80      0.80      0.80      4221



In [ ]:
# 3. Model 7 - rUs with KNN
###
from sklearn import neighbors
import time
start_time = time.time()

# Instantiate classifier
clf7_name = "rUs with KNN"
clf7 = neighbors.KNeighborsClassifier(n_neighbors = 7, metric = 'minkowski')

# Train the model on training data
clf7.fit(X_ru, y_ru)

# Make predictions on test set
y_pred = clf7.predict(X_test)

# Calc accuracys
clf7_score = clf7.score(X_test, y_test)

# Calc mean unweighted F1 Score in all classes
clf7_f1 = f1_score(y_test, y_pred, average = "macro")

# Measure time
model7_time = (time.time() - start_time)/60

In [ ]:
# Show Results
###

# Modelling Time
print("Model 7: --- %s minutes ---" % model7_time)

# Score and F1-Score
print("The score is:", clf7_score)
print("The mean F1-Score (unweighted) is:", clf7_f1)

# Show Confusion Matrix
cm7 = pd.crosstab(y_test, y_pred, rownames = ['Realised Class'], colnames = ['Predicted Class'])
display(cm7)

# Show classification report
model7_cr = classification_report(y_test, y_pred)
print(model7_cr)

Model 7: --- 0.20916967391967772 minutes ---
The score is: 0.7048092868988391
The mean F1-Score (unweighted) is: 0.7100589618928661


Predicted Class,0,1,2,3
Realised Class,,,,
0,1466,235,229,173
1,92,458,118,13
2,173,173,788,31
3,1,4,4,263


              precision    recall  f1-score   support

           0       0.85      0.70      0.76      2103
           1       0.53      0.67      0.59       681
           2       0.69      0.68      0.68      1165
           3       0.55      0.97      0.70       272

    accuracy                           0.70      4221
   macro avg       0.65      0.75      0.68      4221
weighted avg       0.73      0.70      0.71      4221



In [ ]:
# 4.2 Oversampling
###

# apply random over sampler
rOs = RandomOverSampler()

# Create new oversampled data sets
X_ro, y_ro = rOs.fit_resample(X_train, y_train)

# Show number of classes in resulting sample
print("Classes starting data:", dict(y_train.value_counts()))
print("Classes oversampled:", dict(pd.Series(y_ro).value_counts()))



Classes starting data: {0: 8088, 2: 4847, 1: 2883, 3: 1066}
Classes oversampled: {1: 8088, 0: 8088, 2: 8088, 3: 8088}


In [ ]:
# Model 8 - rOs with KNN
###
import time
start_time = time.time()

# Instantiate classifier
clf8_name = "rOs with KNN"
clf8 = neighbors.KNeighborsClassifier(n_neighbors = 7, metric = 'minkowski')

# Train the model on training data
clf8.fit(X_ro, y_ro)

# Make predictions on test set
y_pred = clf8.predict(X_test)

# Calc accuracys
clf8_score = clf8.score(X_test, y_test)

# Calc mean unweighted F1 Score in all classes
clf8_f1 = f1_score(y_test, y_pred, average = "macro")

# Measure time
model8_time = (time.time() - start_time)/60


In [ ]:
# Show Results
###

# Modelling Time
print("Model 8: --- %s minutes ---" % model8_time)

# Score and F1-Score
print("The score is:", clf8_score)
print("The mean F1-Score (unweighted) is:", clf8_f1)

# Show Confusion Matrix
cm8 = pd.crosstab(y_test, y_pred, rownames = ['Realised Class'], colnames = ['Predicted Class'])
display(cm8)

# Show classification report
model8_cr = classification_report(y_test, y_pred)
print(model8_cr)

Model 8: --- 1.4875235954920452 minutes ---
The score is: 0.7393982468609335
The mean F1-Score (unweighted) is: 0.7433672513359342


Predicted Class,0,1,2,3
Realised Class,,,,
0,1560,202,241,100
1,64,493,112,12
2,166,166,810,23
3,4,5,5,258


              precision    recall  f1-score   support

           0       0.87      0.74      0.80      2103
           1       0.57      0.72      0.64       681
           2       0.69      0.70      0.69      1165
           3       0.66      0.95      0.78       272

    accuracy                           0.74      4221
   macro avg       0.70      0.78      0.73      4221
weighted avg       0.76      0.74      0.74      4221



In [ ]:
# Model 9 - rOs with SVM
###
clf9_name = "rOs with SVM"
clf9 = svm.SVC(gamma = 0.01, kernel = "poly")

# Train the model on the randomoversampler training sets
clf9.fit(X_ro, y_ro)

# Make predictions
y_pred = clf9.predict(X_test)

# Calc accuracys
clf9_score = clf9.score(X_test, y_test)

# Calc mean unweighted F1 Score in all classes
clf9_f1 = f1_score(y_test, y_pred, average = "macro")

# Measure time
model9_time = (time.time() - start_time)/60

In [ ]:
# Show Results
###

# Modelling Time
print("Model 9: --- %s minutes ---" % model9_time)

# Score and F1-Score
print("The score is:", clf9_score)
print("The mean F1-Score (unweighted) is:", clf9_f1)

# Show Confusion Matrix
cm9 = pd.crosstab(y_test, y_pred, rownames = ['Realised Class'], colnames = ['Predicted Class'])
display(cm9)

# Show classification report
model9_cr = classification_report(y_test, y_pred)
print(model9_cr)

Model 9: --- 53.69916494290034 minutes ---
The score is: 0.798862828713575
The mean F1-Score (unweighted) is: 0.798602859354799


Predicted Class,0,1,2,3
Realised Class,,,,
0,1781,100,202,20
1,123,474,83,1
2,206,77,878,4
3,19,8,6,239


              precision    recall  f1-score   support

           0       0.84      0.85      0.84      2103
           1       0.72      0.70      0.71       681
           2       0.75      0.75      0.75      1165
           3       0.91      0.88      0.89       272

    accuracy                           0.80      4221
   macro avg       0.80      0.79      0.80      4221
weighted avg       0.80      0.80      0.80      4221

